In [ ]:
# pre-tokenization

import os
from typing import BinaryIO
import multiprocessing 
from collections import Counter
import regex as re



In [ ]:
freq = Counter()

PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""



def find_chunk_boundaries(
    file: BinaryIO,
    desired_num_chunks: int,
    split_special_token: bytes,
) -> list[int]:
    
    """
    Chunk the file into parts that can be counted independently.
    May return fewer chunks if the boundaries end up overlapping.
    """
    assert isinstance(split_special_token, bytes), "Must represent special token as a bytestring"

    # Get total file size in bytes
    file.seek(0, os.SEEK_END) # (offset and from where) moves the cursor to the end.
    file_size = file.tell() # cursor is at the end, tell() returns where the cursor is at (int) 
    file.seek(0) # move the cursor back to the end 

    chunk_size = file_size // desired_num_chunks

    # Initial guesses for chunk boundary locations, uniformly spaced
    # Chunks start on previous index, don't include last index

    chunk_boundaries = [i * chunk_size for i in range(desired_num_chunks + 1)] # chunk_size * (0...desired_num_chunks + 1)
    chunk_boundaries[-1] = file_size # last boundary at the end of the file.

    mini_chunk_size = 4096  # Read ahead by 4k bytes at a time

    for bi in range(1, len(chunk_boundaries) - 1):
        initial_position = chunk_boundaries[bi]
        file.seek(initial_position)  # Start at boundary guess
        while True:
            mini_chunk = file.read(mini_chunk_size)  # Read a mini chunk

            # If EOF, this boundary should be at the end of the file
            if mini_chunk == b"":
                chunk_boundaries[bi] = file_size
                break

            # Find the special token in the mini chunk
            found_at = mini_chunk.find(split_special_token)
            if found_at != -1:
                chunk_boundaries[bi] = initial_position + found_at
                break
            initial_position += mini_chunk_size

    # Make sure all boundaries are unique, but might be fewer than desired_num_chunks
    return sorted(set(chunk_boundaries))



In [ ]:
def pre_tokn(job:tuple):
        input_path, start, end, special_token = job
        with open(input_path, "rb") as f:
            f.seek(start)
            chunk = f.read(end - start)
            # main pre-tokenization process
            # re.finditer
            docs = re.split(special_token,chunk)
            for doc in docs:
               for match in re.finditer(PAT,doc):
                   freq += Counter(match)
                   print(freq)
                   
                   
               


def bpe_tokn(input_path:str, vocab_size:int, special_token:list[str]):
    ## chunking for parallel processing.
    with open(input_path, "rb") as f:
        with multiprocessing.Pool(processes=4) as pool:
            num_processes = 4
            boundaries = find_chunk_boundaries(f, num_processes, special_token)
            jobs = [(input_path,start, end, special_token) for start, end in zip(boundaries[:-1], boundaries[1:])]
            chunks = pool.map(pre_tokn, jobs)

    return None #vocab, merges # dict[int,byte] and list[tuple[bytes,bytes]]


input_path = "/Users/anshmittal/projects/assignment1-basics/data/TinyStoriesV2-GPT4-valid.txt"

In [ ]:
# special_token = b"<|endoftext|>"
# bpe_tokn(input_path,32,special_token)

In [ ]:
print(special_token.decode("UTF-8"))

In [ ]:
c = Counter(special_token)

In [ ]:
print(c)

In [ ]:
b = Counter(special_token)

In [ ]:
b

In [ ]:
c

In [ ]:
c.update(b)

In [ ]:
c

In [ ]:
doc = "Once upon a time there was a little girl named Emily who loved laundry. Every day after school, she'd run over to her mom's washer and dryer and check on her laundry. She loved the way the clothes smelled and the way they felt after they were fresh and fluffy. It was a very interesting thing for Emily to do.
One day, Emily's mother came out of the kitchen and spotted her daughter playing with the laundry. "Emily, what are you doing?" her mother asked. Emily smiled and said, "I love laundry, mommy. It's so interesting!"
Her mom laughed and said, "That's nice honey, but why don't you go play with your toys instead?" So Emily went off to play with her toys, but still she loved doing laundry. Every day she would sneak into the laundry room and help her mom with the laundry.
Emily's mom was amazed at how much her daughter loved doing laundry! She was so proud of Emily and made sure to give her extra hugs because of it. Emily was so happy that her mom was showing her so much love.
From then on, Emily was known as the Laundry Queen! Even when other kids were playing, Emily was still doing her favorite thing - laundry. Her friends thought it was very interesting and wanted to join in. So they'd all help Emily with the laundry, and it made them very happy. 
The end.


Once upon a time, in a small house, there lived a little girl named Mia and her mom. Mia loved to play with her toys and have fun with her friends. One day, Mia felt very tired after playing all day.
Mia said to her mom, "Mommy, I am so tired. My legs hurt." Her mom looked at Mia and thought of a way to help her feel better. She suggested, "Mia, let me give you a small massage to help your legs feel better."
Mia agreed and said, "Okay, Mommy." Her mom started to massage her legs gently. As her mom massaged her, Mia felt her legs getting better and better. She smiled at her mom and said, "Thank you, Mommy. My legs feel so much better now."
From that day on, whenever Mia felt tired or her legs hurt, her mom would give her a small massage to help her feel better. Mia and her mom were always happy and loved spending time together.


Once upon a time, in a hot and sandy land, there was a little camel named Cally. Cally lived near an oasis, a place with water and trees. She loved to play there with her friends.
One day, a fierce lion came to the oasis. He was very big and scary. The lion roared and demanded, "Give me all your food!" Cally and her friends were scared, but they wanted to save their oasis.
Cally had an idea. She told her friends to make a big, loud noise. They stomped their feet and yelled together. The lion was scared of the noise and ran away. Cally and her friends saved the oasis and were very happy.

One day, a little boy named Tim went for a walk in the wild woods. He saw a small bird with a hurt wing. The bird had a bandage on it. Tim felt sad for the bird and wanted to help.
He said, "Hi bird, I can help you. Let's turn around and go to my house." The bird nodded and Tim carefully picked it up. They walked back to his house together.
At Tim's house, his mom saw the bird and was surprised. She said, "Oh no! That is not a real bird. It is a toy!" Tim looked at the toy bird and laughed. He had made a new friend, even if it was just a toy."

In [ ]:
doc = '''
Once upon a time there was a little girl named Emily who loved laundry. Every day after school, she'd run over to her mom's washer and dryer and check on her laundry. She loved the way the clothes smelled and the way they felt after they were fresh and fluffy. It was a very interesting thing for Emily to do.
One day, Emily's mother came out of the kitchen and spotted her daughter playing with the laundry. "Emily, what are you doing?" her mother asked. Emily smiled and said, "I love laundry, mommy. It's so interesting!"
Her mom laughed and said, "That's nice honey, but why don't you go play with your toys instead?" So Emily went off to play with her toys, but still she loved doing laundry. Every day she would sneak into the laundry room and help her mom with the laundry.
Emily's mom was amazed at how much her daughter loved doing laundry! She was so proud of Emily and made sure to give her extra hugs because of it. Emily was so happy that her mom was showing her so much love.
From then on, Emily was known as the Laundry Queen! Even when other kids were playing, Emily was still doing her favorite thing - laundry. Her friends thought it was very interesting and wanted to join in. So they'd all help Emily with the laundry, and it made them very happy. 
The end.


Once upon a time, in a small house, there lived a little girl named Mia and her mom. Mia loved to play with her toys and have fun with her friends. One day, Mia felt very tired after playing all day.
Mia said to her mom, "Mommy, I am so tired. My legs hurt." Her mom looked at Mia and thought of a way to help her feel better. She suggested, "Mia, let me give you a small massage to help your legs feel better."
Mia agreed and said, "Okay, Mommy." Her mom started to massage her legs gently. As her mom massaged her, Mia felt her legs getting better and better. She smiled at her mom and said, "Thank you, Mommy. My legs feel so much better now."
From that day on, whenever Mia felt tired or her legs hurt, her mom would give her a small massage to help her feel better. Mia and her mom were always happy and loved spending time together.


Once upon a time, in a hot and sandy land, there was a little camel named Cally. Cally lived near an oasis, a place with water and trees. She loved to play there with her friends.
One day, a fierce lion came to the oasis. He was very big and scary. The lion roared and demanded, "Give me all your food!" Cally and her friends were scared, but they wanted to save their oasis.
Cally had an idea. She told her friends to make a big, loud noise. They stomped their feet and yelled together. The lion was scared of the noise and ran away. Cally and her friends saved the oasis and were very happy.

One day, a little boy named Tim went for a walk in the wild woods. He saw a small bird with a hurt wing. The bird had a bandage on it. Tim felt sad for the bird and wanted to help.
He said, "Hi bird, I can help you. Let's turn around and go to my house." The bird nodded and Tim carefully picked it up. They walked back to his house together.
At Tim's house, his mom saw the bird and was surprised. She said, "Oh no! That is not a real bird. It is a toy!" Tim looked at the toy bird and laughed. He had made a new friend, even if it was just a toy.
'''


In [ ]:
len(doc)

In [ ]:
PAT
counter = Counter()
c = Counter()

In [ ]:
for match in re.finditer(PAT,doc):
    pretoken = match.group()
    counter[pretoken.encode("utf-8")] += 1

In [ ]:
for k,v in counter.items():
    k = tuple(bytes([b]) for b in k)
    c[k] = v
    
    

In [ ]:
c

In [ ]:
a = "hi".encode()
b = list(a)


In [ ]:
b

In [ ]:
counter.items()
counter[b'\n']


In [ ]:
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""


In [ ]:
i = re.finditer(PAT,doc)

In [ ]:
i

In [ ]:
for _ in i:
    

In [ ]:
special_token.decode("utf-8")

In [ ]:
doc += "<|endoftext|>"

In [ ]:
doc = '''
'\nOnce upon a time there was a little girl named Emily who loved laundry. <|endoftext|> Every day after school, she\'d run over to her mom\'s washer and dryer and check on her laundry. She loved the way the clothes smelled and the way they felt after they were fresh and fluffy. It was a very interesting thing for Emily to do.\nOne day, Emily\'s mother came out of the kitchen and spotted her daughter playing with the laundry. "Emily, what are you doing?" her mother asked. Emily smiled and said, "I love laundry, mommy. It\'s so interesting!"\nHer mom laughed and said, "That\'s nice honey, but why don\'t you go play with your toys instead?" So Emily went off to play with her toys, but still she loved doing laundry. Every day she would sneak into the laundry room and help her mom with the laundry.\nEmily\'s mom was amazed at how much her daughter loved doing laundry! She was so proud of Emily and made sure to give her extra hugs because of it. Emily was so happy that her mom was showing her so much love.\nFrom then on, Emily was known as the Laundry Queen! Even when other kids were playing, Emily was still doing her favorite thing - laundry. Her friends thought it was very interesting and wanted to join in. So they\'d all help Emily with the laundry, and it made them very happy. \nThe end.\n\n\nOnce upon a time, in a small house, there lived a little girl named Mia and her mom. Mia loved to play with her toys and have fun with her friends. One day, Mia felt very tired after playing all day.\nMia said to her mom, "Mommy, I am so tired. My legs hurt." Her mom looked at Mia and thought of a way to help her feel better. She suggested, "Mia, let me give you a small massage to help your legs feel better."\nMia agreed and said, "Okay, Mommy." Her mom started to massage her legs gently. As her mom massaged her, Mia felt her legs getting better and better. She smiled at her mom and said, "Thank you, Mommy. My legs feel so much better now."\nFrom that day on, whenever Mia felt tired or her legs hurt, her mom would give her a small massage to help her feel better. Mia and her mom were always happy and loved spending time together.\n\n\nOnce upon a time, in a hot and sandy land, there was a little camel named Cally. Cally lived near an oasis, a place with water and trees. She loved to play there with her friends.\nOne day, a fierce lion came to the oasis. He was very big and scary. The lion roared and demanded, "Give me all your food!" Cally and her friends were scared, but they wanted to save their oasis.\nCally had an idea. She told her friends to make a big, loud noise. They stomped their feet and yelled together. The lion was scared of the noise and ran away. Cally and her friends saved the oasis and were very happy.\n\nOne day, a little boy named Tim went for a walk in the wild woods. He saw a small bird with a hurt wing. The bird had a bandage on it. Tim felt sad for the bird and wanted to help.\nHe said, "Hi bird, I can help you. Let\'s turn around and go to my house." The bird nodded and Tim carefully picked it up. They walked back to his house together.\nAt Tim\'s house, his mom saw the bird and was surprised. She said, "Oh no! That is not a real bird. It is a toy!" Tim looked at the toy bird and laughed. He had made a new friend, even if it was just a toy.\n<|endoftext|>'


'''

In [ ]:

special_token_str = special_token.decode("utf-8")
docs = re.split(re.escape(special_token_str), doc)

In [ ]:
docs

In [ ]:
counter

In [ ]:
char_count = {}
for k,v in counter.items():
    k = tuple(bytes([b]) for b in k)
    char_count[k] = v
    

In [ ]:
vocab = {} 
for b in char_count: 
    list(b)
    if len(b) >0:
        for i in range(len(b)-1):
            if b[i],b[i+1] in vocab:
            pass
            

In [ ]:
char_count

In [ ]:
a = "Z".encode() 



In [ ]:
a

In [ ]:
list(a)
i= 65

In [ ]:
bytes([i])

In [ ]:
vocab = {i:bytes([i]) for i in range(255)}

In [ ]:


vocab[256] = special_token